In [6]:
import pandas as pd
import re

lineup_df = pd.read_csv("npb_2024_lineup_all.csv")

def parse_lineup_date(x):
    s = str(x).strip()
    s = s.replace("月", "/").replace("日", "")
    s = re.sub(r"\(.*?\)", "", s)   # (土) など削除
    s = re.sub(r"[^0-9/]", "", s)   # 数字と/以外を削除

    m = re.search(r"(\d{1,2})/(\d{1,2})", s)
    if m:
        month = int(m.group(1))
        day = int(m.group(2))
        return pd.Timestamp(f"2025-{month:02d}-{day:02d}")
    return pd.NaT

lineup_df["日付"] = lineup_df["日付"].apply(parse_lineup_date)

print(lineup_df[["球団", "日付", "4番"]].head())
print(lineup_df["日付"].isna().sum())

   球団         日付     4番
0  阪神 2025-03-29  大山 悠輔
1  阪神 2025-03-30  大山 悠輔
2  阪神 2025-03-31  大山 悠輔
3  阪神 2025-04-02  大山 悠輔
4  阪神 2025-04-03  大山 悠輔
0


In [9]:
import calendar
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

HEADERS = {"User-Agent": "Mozilla/5.0"}

CALENDAR_PAGES = {
    "04": "https://npb.jp/bis/eng/2024/calendar/index_04.html",
    "05": "https://npb.jp/bis/eng/2024/calendar/index_05.html",
    "06": "https://npb.jp/bis/eng/2024/calendar/index_06.html",
    "07": "https://npb.jp/bis/eng/2024/calendar/index_07.html",
    "08": "https://npb.jp/bis/eng/2024/calendar/index_08.html",
    "09": "https://npb.jp/bis/eng/2024/calendar/index_09.html",
    "10": "https://npb.jp/bis/eng/2024/calendar/index_10.html",
}

REVERSE_TEAM_CODE_MAP = {
    "G": "巨人", "T": "阪神", "DB": "DeNA", "C": "広島", "S": "ヤクルト", "D": "中日",
    "H": "ソフトバンク", "F": "日本ハム", "B": "オリックス", "M": "ロッテ", "E": "楽天", "L": "西武"
}

def parse_score_line(line: str):
    line = re.sub(r"\s+", " ", line.strip())
    m = re.fullmatch(r"([A-Z]+) (\d+) - (\d+) ([A-Z]+)", line)
    if not m:
        return None
    return {
        "team1": m.group(1),
        "score1": int(m.group(2)),
        "score2": int(m.group(3)),
        "team2": m.group(4),
    }

def make_safe_date(year, month, day):
    last_day = calendar.monthrange(year, month)[1]
    if 1 <= day <= last_day:
        return pd.Timestamp(year=year, month=month, day=day)
    return None

def scrape_month_results(page_code: str, url: str) -> pd.DataFrame:
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = r.apparent_encoding

    soup = BeautifulSoup(r.text, "html.parser")
    lines = [x.strip() for x in soup.get_text("\n").splitlines() if x.strip()]

    rows = []
    current_date = None

    for line in lines:
        if re.fullmatch(r"\d{1,2}", line):
            day = int(line)

            if page_code == "04":
                # 2024年は 3-4月ページ
                if 29 <= day <= 31:
                    dt = make_safe_date(2024, 3, day)
                else:
                    dt = make_safe_date(2024, 4, day)
            else:
                dt = make_safe_date(2024, int(page_code), day)

            if dt is not None:
                current_date = dt
            continue

        parsed = parse_score_line(line)
        if parsed and current_date is not None:
            rows.append({
                "日付": current_date,
                "team1": parsed["team1"],
                "score1": parsed["score1"],
                "team2": parsed["team2"],
                "score2": parsed["score2"],
                "raw": line,
            })

    return pd.DataFrame(rows)

all_months = []

for page_code, url in CALENDAR_PAGES.items():
    try:
        print(f"取得中: {page_code}")
        mdf = scrape_month_results(page_code, url)
        print(f"  {len(mdf)}試合分")
        if not mdf.empty:
            all_months.append(mdf)
    except Exception as e:
        print(f"[ERROR] {page_code}: {e}")

games_df = pd.concat(all_months, ignore_index=True)

team_game_rows = []
for _, row in games_df.iterrows():
    team_game_rows.append({
        "日付": row["日付"],
        "球団": REVERSE_TEAM_CODE_MAP.get(row["team1"]),
        "得点": row["score1"],
        "raw": row["raw"],
    })
    team_game_rows.append({
        "日付": row["日付"],
        "球団": REVERSE_TEAM_CODE_MAP.get(row["team2"]),
        "得点": row["score2"],
        "raw": row["raw"],
    })

team_scores_df = pd.DataFrame(team_game_rows).dropna(subset=["球団"])

print(team_scores_df.head())
print(team_scores_df["日付"].min(), team_scores_df["日付"].max())

取得中: 04
  157試合分
取得中: 05
  140試合分
取得中: 06
  134試合分
取得中: 07
  126試合分
取得中: 08
  150試合分
取得中: 09
  134試合分
取得中: 10
  38試合分
          日付    球団  得点         raw
0 2024-03-29    巨人   4   G 4 - 0 T
1 2024-03-29    阪神   0   G 4 - 0 T
2 2024-03-29  ヤクルト   7   S 7 - 4 D
3 2024-03-29    中日   4   S 7 - 4 D
4 2024-03-29  DeNA   4  DB 4 - 3 C
2024-03-29 00:00:00 2024-10-31 00:00:00


In [10]:
print("lineup_df sample")
print(lineup_df[["球団", "日付", "4番"]].head())

print("\nteam_scores_df sample")
print(team_scores_df[["球団", "日付", "得点"]].head())

print("\nDeNA lineup dates")
print(lineup_df[lineup_df["球団"] == "DeNA"]["日付"].dropna().head())

print("\nDeNA score dates")
print(team_scores_df[team_scores_df["球団"] == "DeNA"]["日付"].dropna().head())

lineup_df sample
   球団         日付     4番
0  阪神 2025-03-29  大山 悠輔
1  阪神 2025-03-30  大山 悠輔
2  阪神 2025-03-31  大山 悠輔
3  阪神 2025-04-02  大山 悠輔
4  阪神 2025-04-03  大山 悠輔

team_scores_df sample
     球団         日付  得点
0    巨人 2024-03-29   4
1    阪神 2024-03-29   0
2  ヤクルト 2024-03-29   7
3    中日 2024-03-29   4
4  DeNA 2024-03-29   4

DeNA lineup dates
143   2025-03-29
144   2025-03-30
145   2025-03-31
146   2025-04-02
147   2025-04-03
Name: 日付, dtype: datetime64[ns]

DeNA score dates
4    2024-03-29
16   2024-03-30
28   2024-03-31
39   2024-04-02
51   2024-04-03
Name: 日付, dtype: datetime64[ns]


In [12]:
import pandas as pd
import re

# 1. lineupデータ読み込み
lineup_df = pd.read_csv("npb_2024_lineup_all.csv")

def parse_lineup_date(x):
    s = str(x).strip()
    s = s.replace("月", "/").replace("日", "")
    s = re.sub(r"\(.*?\)", "", s)
    s = re.sub(r"[^0-9/]", "", s)

    m = re.search(r"(\d{1,2})/(\d{1,2})", s)
    if m:
        month = int(m.group(1))
        day = int(m.group(2))
        return pd.Timestamp(f"2024-{month:02d}-{day:02d}")
    return pd.NaT

lineup_df["日付"] = lineup_df["日付"].apply(parse_lineup_date)
lineup_df["4番"] = lineup_df["4番"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 2. 代表4番打者一覧
main_fourth_df = pd.DataFrame({
    "球団": ["DeNA", "オリックス", "ソフトバンク", "ヤクルト", "ロッテ", "中日",
           "巨人", "広島", "日本ハム", "楽天", "西武", "阪神"],
    "選手名": ["ここを2024年の選手名に置き換える"] * 12
})

# ↑ ここは2024年版の代表4番打者一覧に差し替えてください
# もしすでにCSVがあるなら、次のように読み込む方が楽です
# main_fourth_df = pd.read_csv("npb_2024_main_fourth_batters.csv")
# main_fourth_df = main_fourth_df.rename(columns={"4番": "選手名"})

main_fourth_df["選手名"] = main_fourth_df["選手名"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

# 3. team_scores_df はすでに作成済み前提
# 念のため日付型をそろえる
team_scores_df["日付"] = pd.to_datetime(team_scores_df["日付"], errors="coerce")

# 4. 結合して player_game_scores_df を作る
merged_list = []

for _, r in main_fourth_df.iterrows():
    team = r["球団"]
    player = r["選手名"]

    subset = lineup_df[
        (lineup_df["球団"] == team) &
        (lineup_df["4番"] == player)
    ][["球団", "日付", "4番"]].copy()

    subset = subset.rename(columns={"4番": "選手名"})

    subset = subset.merge(
        team_scores_df[["球団", "日付", "得点"]],
        on=["球団", "日付"],
        how="left"
    )

    merged_list.append(subset)

player_game_scores_df = pd.concat(merged_list, ignore_index=True)

print(player_game_scores_df.head())
print("得点欠損数:", player_game_scores_df["得点"].isna().sum())

Empty DataFrame
Columns: [球団, 日付, 選手名, 得点]
Index: []
得点欠損数: 0


In [13]:
summary_df = (
    player_game_scores_df
    .groupby(["球団", "選手名"], as_index=False)
    .agg({"得点": ["count", "sum", "mean"]})
)

summary_df.columns = ["球団", "選手名", "4番試合数", "合計得点", "平均得点"]
summary_df["平均得点"] = summary_df["平均得点"].round(3)

In [14]:
print(lineup_df["日付"].head(20).tolist())

[Timestamp('2024-03-29 00:00:00'), Timestamp('2024-03-30 00:00:00'), Timestamp('2024-03-31 00:00:00'), Timestamp('2024-04-02 00:00:00'), Timestamp('2024-04-03 00:00:00'), Timestamp('2024-04-04 00:00:00'), Timestamp('2024-04-05 00:00:00'), Timestamp('2024-04-06 00:00:00'), Timestamp('2024-04-07 00:00:00'), Timestamp('2024-04-09 00:00:00'), Timestamp('2024-04-10 00:00:00'), Timestamp('2024-04-11 00:00:00'), Timestamp('2024-04-12 00:00:00'), Timestamp('2024-04-13 00:00:00'), Timestamp('2024-04-14 00:00:00'), Timestamp('2024-04-16 00:00:00'), Timestamp('2024-04-17 00:00:00'), Timestamp('2024-04-18 00:00:00'), Timestamp('2024-04-19 00:00:00'), Timestamp('2024-04-20 00:00:00')]


In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time

# =========================
# 0. 既存ファイル
# =========================
lineup_df = pd.read_csv("npb_2024_lineup_all.csv")
lineup_df["日付"] = pd.to_datetime(lineup_df["日付"], errors="coerce")
lineup_df["4番"] = lineup_df["4番"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

main_fourth_df = pd.DataFrame({
    "球団": ["DeNA", "オリックス", "ソフトバンク", "ヤクルト", "ロッテ", "中日",
           "巨人", "広島", "日本ハム", "楽天", "西武", "阪神"],
    "選手名": ["牧 秀悟", "森 友哉", "山川 穂高", "村上 宗隆", "ソト", "細川 成也",
            "岡本 和真", "小園 海斗", "マルティネス", "浅村 栄斗", "佐藤 龍世", "大山 悠輔"]
})
main_fourth_df["選手名"] = main_fourth_df["選手名"].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()

TEAM_CODE_MAP = {
    "巨人": "G", "阪神": "T", "DeNA": "DB", "広島": "C", "ヤクルト": "S", "中日": "D",
    "ソフトバンク": "H", "日本ハム": "F", "オリックス": "B", "ロッテ": "M", "楽天": "E", "西武": "L",
}
REVERSE_TEAM_CODE_MAP = {v: k for k, v in TEAM_CODE_MAP.items()}

HEADERS = {"User-Agent": "Mozilla/5.0"}

# 2024年は 04=Mar-Apr, 05=May, ..., 10=Oct
CALENDAR_PAGES = {
    "04": "https://npb.jp/bis/eng/2024/calendar/index_04.html",
    "05": "https://npb.jp/bis/eng/2024/calendar/index_05.html",
    "06": "https://npb.jp/bis/eng/2024/calendar/index_06.html",
    "07": "https://npb.jp/bis/eng/2024/calendar/index_07.html",
    "08": "https://npb.jp/bis/eng/2024/calendar/index_08.html",
    "09": "https://npb.jp/bis/eng/2024/calendar/index_09.html",
    "10": "https://npb.jp/bis/eng/2024/calendar/index_10.html",
}

def parse_score_line(line: str):
    line = re.sub(r"\s+", " ", line.strip())
    m = re.fullmatch(r"([A-Z]+) (\d+) - (\d+) ([A-Z]+)", line)
    if not m:
        return None
    return {
        "team1": m.group(1),
        "score1": int(m.group(2)),
        "score2": int(m.group(3)),
        "team2": m.group(4),
    }

def scrape_month_results(page_code: str, url: str) -> pd.DataFrame:
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    r.encoding = r.apparent_encoding

    soup = BeautifulSoup(r.text, "html.parser")
    lines = [x.strip() for x in soup.get_text("\n").splitlines() if x.strip()]

    rows = []
    current_date = None

    # 04ページだけは Mar/Apr 混在なので month を切り替える
    current_month = "04"

    for line in lines:
        # 月の切り替え検知（3・4月ページ用）
        if "Calendar (March - April 2024)" in line:
            current_month = "03"

        # 日だけの行（例: 28, 29, 1, 2 ...）
        if re.fullmatch(r"\d{1,2}", line):
            day = int(line)

            if page_code == "04":
                # 3月28〜31は3月、それ以外は4月として扱う
                if day >= 28 and current_month == "03":
                    month = "03"
                else:
                    month = "04"
                    current_month = "04"
            else:
                month = page_code

            current_date = pd.Timestamp(f"2024-{month}-{day:02d}")
            continue

        parsed = parse_score_line(line)
        if parsed and current_date is not None:
            rows.append({
                "日付": current_date,
                "team1": parsed["team1"],
                "score1": parsed["score1"],
                "team2": parsed["team2"],
                "score2": parsed["score2"],
                "raw": line,
            })

    return pd.DataFrame(rows)

all_months = []

for page_code, url in CALENDAR_PAGES.items():
    try:
        print(f"取得中: {page_code} -> {url}")
        mdf = scrape_month_results(page_code, url)
        print(f"  {len(mdf)}試合分")
        if not mdf.empty:
            all_months.append(mdf)
        time.sleep(1)
    except Exception as e:
        print(f"[ERROR] {page_code}: {e}")

if not all_months:
    raise RuntimeError("月別データを1件も取得できませんでした。URLかHTML構造を確認してください。")

games_df = pd.concat(all_months, ignore_index=True)

# 1試合1行 -> 1チーム1行
team_game_rows = []
for _, row in games_df.iterrows():
    team_game_rows.append({
        "日付": row["日付"],
        "球団": REVERSE_TEAM_CODE_MAP.get(row["team1"]),
        "得点": row["score1"],
        "raw": row["raw"],
    })
    team_game_rows.append({
        "日付": row["日付"],
        "球団": REVERSE_TEAM_CODE_MAP.get(row["team2"]),
        "得点": row["score2"],
        "raw": row["raw"],
    })

team_scores_df = pd.DataFrame(team_game_rows).dropna(subset=["球団"])

merged_list = []
for _, r in main_fourth_df.iterrows():
    team = r["球団"]
    player = r["選手名"]

    subset = lineup_df[(lineup_df["球団"] == team) & (lineup_df["4番"] == player)][["球団", "日付", "4番"]].copy()
    subset = subset.rename(columns={"4番": "選手名"})

    subset = subset.merge(
        team_scores_df[["球団", "日付", "得点", "raw"]],
        on=["球団", "日付"],
        how="left"
    )
    merged_list.append(subset)

player_game_scores_df = pd.concat(merged_list, ignore_index=True)

summary_df = (
    player_game_scores_df
    .groupby(["球団", "選手名"], as_index=False)
    .agg({"得点": ["count", "sum", "mean"]})
)

summary_df.columns = ["球団", "選手名", "4番試合数", "合計得点", "平均得点"]
summary_df["平均得点"] = summary_df["平均得点"].round(3)

print(summary_df.sort_values("平均得点", ascending=False))

player_game_scores_df.to_csv("npb_2024_main_fourth_batter_game_scores.csv", index=False, encoding="utf-8-sig")
summary_df.to_csv("npb_2024_main_fourth_batter_avg_runs_when_batting_4th.csv", index=False, encoding="utf-8-sig")
print("保存完了")

/var/folders/7z/j354sfjn68qdc68lznr2tqdh0000gn/T/ipykernel_65495/3993981281.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  lineup_df["日付"] = pd.to_datetime(lineup_df["日付"], errors="coerce")


取得中: 04 -> https://npb.jp/bis/eng/2024/calendar/index_04.html
[ERROR] 04: day is out of range for month: 2024-04-31
取得中: 05 -> https://npb.jp/bis/eng/2024/calendar/index_05.html
  140試合分
取得中: 06 -> https://npb.jp/bis/eng/2024/calendar/index_06.html
[ERROR] 06: day is out of range for month: 2024-06-31
取得中: 07 -> https://npb.jp/bis/eng/2024/calendar/index_07.html
  126試合分
取得中: 08 -> https://npb.jp/bis/eng/2024/calendar/index_08.html
  150試合分
取得中: 09 -> https://npb.jp/bis/eng/2024/calendar/index_09.html
[ERROR] 09: day is out of range for month: 2024-09-31
取得中: 10 -> https://npb.jp/bis/eng/2024/calendar/index_10.html
  38試合分
        球団     選手名  4番試合数  合計得点  平均得点
0     DeNA    牧 秀悟      0   0.0   NaN
1    オリックス    森 友哉      0   0.0   NaN
2   ソフトバンク   山川 穂高      0   0.0   NaN
3     ヤクルト   村上 宗隆      0   0.0   NaN
4      ロッテ      ソト      0   0.0   NaN
5       中日   細川 成也      0   0.0   NaN
6       巨人   岡本 和真      0   0.0   NaN
7       広島   小園 海斗      0   0.0   NaN
8     日本ハム  マルティネス      0  